In [31]:
import torch
import torch.nn as nn
import torch.functional as F

In [33]:
class GATReimplementation(nn.Module):
    def __init__(self, in_channel: int, out_channel: int, dropout: float):
        super().__init__()

        self.in_channel = in_channel
        self.out_channel = out_channel

        self.W = nn.Parameter(torch.zeros(size=(in_channel, out_channel)))
        nn.init.xavier_uniform_(self.W.data, gain=1.141)
        self.a = nn.Parameter(torch.zeros(size=(2*out_channel, 1)))
        nn.init.xavier_uniform_(self.a.data, gain=1.141)
        self.dropout = nn.Dropout(p=dropout)
        self.softmax = nn.Softmax(dim=1)
        self.leakyrelu = nn.LeakyReLU()
    
    def forward(self, x: torch.Tensor, adj: torch.Tensor):

        h = x @ self.W
        num_nodes = h.shape[0]
        h_concat = torch.cat([h.repeat(1, num_nodes).view(-1, self.out_channel), h.repeat(num_nodes, 1)], dim=1).view(num_nodes, num_nodes, -1)
        a = self.leakyrelu((h_concat @ self.a)).squeeze(2)
        zero_vec = -9e15 * torch.ones_like(a)
        attention = torch.where(adj > 0, a, zero_vec)
        attention = self.softmax(attention)
        attention = self.dropout(attention)
        return attention @ h

In [29]:
num_nodes = 3
in_channel = 6
out_channel = 10
dropout = 0.5
x_test = torch.randn(size=(num_nodes, in_channel))

model = GATReimplementation(in_channel, out_channel, dropout)
model(x_test)

tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]],
       grad_fn=<ViewBackward0>)

In [ ]:

torch.where()